In [3]:
"""
win_060_stacker_gpu.py
GPU (CUDA) accelerated, decoupled multi-tower log-odds stacking.
"""
import os
import gc
import time
import warnings
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

DATA_DIR = "."
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GLOBAL_PRIOR = 0.1805
EPS = 1e-4

def clean_str(s):
    return s.astype(str).str.strip().str.lower().fillna("missing")

def clean_category(s):
    c = clean_str(s)
    mapping = {
        "home decor": "home_decor", "home_decoration": "home_decor",
        "kitchen dining": "kitchen_dining", "sports leisure": "sports_leisure",
        "sports & leisure": "sports_leisure", "health beauty": "health_beauty",
        "fashion accessories": "fashion_accessories", "electronics accessories": "electronics_accessories",
        "toys gifts": "toys_gifts", "garden diy": "garden_diy", "office supplies": "office_supplies"
    }
    return c.replace(mapping)

def to_log_odds_delta(prob, prior=GLOBAL_PRIOR):
    prob = np.clip(prob, EPS, 1.0 - EPS)
    prior = np.clip(prior, EPS, 1.0 - EPS)
    return (logit(prob) - logit(prior)).astype(np.float32)

print("=" * 80)
print(">>> [1/6] LOADING DATA, HYGIENE & POINT-IN-TIME MERGES...")
t0 = time.time()

cust_df = pd.read_csv(f"{DATA_DIR}/customers.csv")
cust_df["region_clean"] = clean_str(cust_df["region"])
cust_df["customer_type"] = clean_str(cust_df["customer_type"])
cust_df["loyalty_tier"] = clean_str(cust_df["loyalty_tier"])
cust_df["age_band"] = clean_str(cust_df["age_band"])
cust_df["preferred_contact"] = clean_str(cust_df["preferred_contact"])
cust_df["newsletter_opt_in"] = cust_df["newsletter_opt_in"].fillna(0).astype(np.int8)
cust_df["record_updated_at"] = pd.to_datetime(cust_df["record_updated_at"])
cust_df["signup_date"] = pd.to_datetime(cust_df["signup_date"])
cust_df = cust_df.sort_values(by="record_updated_at").reset_index(drop=True)

prod_df = pd.read_csv(f"{DATA_DIR}/products.csv")
prod_df["category_clean"] = clean_category(prod_df["category"])
prod_df["prod_volume"] = (prod_df["length_cm"] * prod_df["width_cm"] * prod_df["height_cm"]).fillna(0).astype(np.float32)
prod_df["prod_density"] = (prod_df["weight_kg"] / (prod_df["prod_volume"] + 1e-4)).fillna(0).astype(np.float32)
prod_df["packaging_type"] = clean_str(prod_df["packaging_type"])
prod_df["color"] = clean_str(prod_df["color"])
prod_df["review_count"] = prod_df["review_count"].fillna(0).astype(np.float32)
prod_df["launch_date"] = pd.to_datetime(prod_df["launch_date"])
prod_cols = ["product_id", "category_clean", "list_price", "weight_kg", "prod_volume", "prod_density",
             "supplier_code", "launch_date", "packaging_type", "color", "review_count"]
prod_df = prod_df[prod_cols]

store_df = pd.read_csv(f"{DATA_DIR}/stores.csv")
store_df["region_clean_store"] = clean_str(store_df["region"])
store_df["store_channel"] = clean_str(store_df["channel"])
store_df["format"] = clean_str(store_df["format"])
store_df["size_sqm"] = store_df["size_sqm"].fillna(store_df["size_sqm"].median()).astype(np.float32)
store_df["opened_date"] = pd.to_datetime(store_df["opened_date"])
store_cols = ["store_id", "region_clean_store", "store_channel", "format", "seller_rating",
              "manager_tenure_years", "size_sqm", "opened_date"]
store_df = store_df[store_cols]

calendar_df = pd.read_csv(f"{DATA_DIR}/calendar.csv")
calendar_df["week_start"] = pd.to_datetime(calendar_df["week_start"])
for c in ["cpi", "fuel_price", "temperature_f", "unemployment", "fuel_surcharge"]:
    calendar_df[c] = calendar_df[c].astype(np.float32)

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
train_df["is_test"] = 0
test_df["is_test"] = 1
test_df["returned"] = np.nan
df = pd.concat([train_df, test_df], ignore_index=True)
del train_df, test_df
gc.collect()

drop_noise_cols = ["promo_channel_code", "picker_shift", "warehouse_zone", "device_type",
                   "tz_hint", "coupon_code", "ship_mode", "gift_wrap"]
for col in drop_noise_cols:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

df["order_ts"] = pd.to_datetime(df["order_ts"])
df["year"] = df["order_ts"].dt.year.astype(np.int16)
df["month"] = df["order_ts"].dt.month.astype(np.int8)
df["year_month"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)
df["month_idx"] = ((df["year"] - 2023) * 12 + df["month"]).astype(np.int8)
df["hour"] = df["order_ts"].dt.hour.astype(np.int8)
df["dayofweek"] = df["order_ts"].dt.dayofweek.astype(np.int8)
df["is_weekend"] = (df["dayofweek"] >= 5).astype(np.int8)

# Point-in-time merge_asof customers
df = df.sort_values(by="order_ts").reset_index(drop=True)
cust_cols = ["customer_id", "record_updated_at", "region_clean", "customer_type", "loyalty_tier",
             "signup_date", "age_band", "preferred_contact", "newsletter_opt_in"]
df = pd.merge_asof(
    df,
    cust_df[cust_cols],
    by="customer_id",
    left_on="order_ts",
    right_on="record_updated_at",
    direction="backward"
)
missing_cust = df["region_clean"].isna()
if missing_cust.sum() > 0:
    cust_first = cust_df.drop_duplicates(subset=["customer_id"], keep="first")
    drop_cols = ["record_updated_at", "region_clean", "customer_type", "loyalty_tier", "signup_date",
                 "age_band", "preferred_contact", "newsletter_opt_in"]
    df_missing = df[missing_cust].drop(columns=drop_cols)
    df_missing = df_missing.merge(cust_first[["customer_id"] + drop_cols], on="customer_id", how="left")
    df.loc[missing_cust, drop_cols] = df_missing[drop_cols].values
    del cust_first, df_missing
    gc.collect()
del cust_df
gc.collect()

df = df.merge(prod_df, on="product_id", how="left")
df = df.merge(store_df, on="store_id", how="left")
del prod_df, store_df
gc.collect()

# Calendar merge_asof by week_start
df = df.sort_values(by="order_ts").reset_index(drop=True)
df = pd.merge_asof(
    df,
    calendar_df.sort_values("week_start"),
    left_on="order_ts",
    right_on="week_start",
    direction="backward"
)
gc.collect()

# ADVERSARIAL IMMUNIZATION
print(">>> Computing monthly percentile ranks...")
df["raw_cust_tenure"] = ((df["order_ts"] - df["signup_date"]).dt.total_seconds() / 86400.0).fillna(0).clip(lower=0).astype(np.float32)
df["raw_prod_tenure"] = ((df["order_ts"] - df["launch_date"]).dt.total_seconds() / 86400.0).fillna(0).clip(lower=0).astype(np.float32)
df["raw_store_tenure"] = ((df["order_ts"] - df["opened_date"]).dt.total_seconds() / 86400.0).fillna(0).clip(lower=0).astype(np.float32)
df["session_clicks"] = df["session_clicks"].fillna(0).astype(np.float32)
df["clicks_per_item"] = (df["session_clicks"] / (df["items_in_order"] + 1e-5)).astype(np.float32)
df["review_count"] = df["review_count"].fillna(0).astype(np.float32)

m_grp = df.groupby("month_idx")
for col_src, col_dst in [("items_in_order", "items_in_order_pct_m"), ("line_no", "line_no_pct_m"),
                         ("quantity", "quantity_pct_m"), ("order_value", "order_value_pct_m"),
                         ("raw_cust_tenure", "cust_tenure_pct_m"), ("raw_prod_tenure", "prod_tenure_pct_m"),
                         ("clicks_per_item", "clicks_per_item_pct_m")]:
    df[col_dst] = m_grp[col_src].rank(pct=True).astype(np.float32)

df["items_in_order_capped"] = np.minimum(df["items_in_order"], 35).astype(np.float32)
df["line_no_capped"] = np.minimum(df["line_no"], 35).astype(np.float32)
df["line_ratio"] = (df["line_no"] / (df["items_in_order"] + 1e-5)).astype(np.float32)

order_grp = df.groupby("order_id")
order_tot_qty = order_grp["quantity"].transform("sum")
df["qty_share_order"] = (df["quantity"] / (order_tot_qty + 1e-5)).astype(np.float32)
net_item_price = df["unit_price"] * df["quantity"] * (1.0 - df["discount"])
df["item_share_order"] = (net_item_price / (df["order_value"] + 1.0)).astype(np.float32)
df["price_ratio"] = (df["unit_price"] / (df["list_price"] + 1e-3)).astype(np.float32)
df["has_discount"] = (df["discount"] > 0.0).astype(np.int8)
df["is_free_freight"] = (df["freight_cost"] <= 0.0).astype(np.int8)
df["freight_ratio"] = (df["freight_cost"] / (df["order_value"] + 1.0)).astype(np.float32)
df["same_region"] = (df["region_clean"] == df["region_clean_store"]).astype(np.int8)

df["order_mean_weight"] = order_grp["weight_kg"].transform("mean").astype(np.float32)
df["order_max_weight"] = order_grp["weight_kg"].transform("max").astype(np.float32)
order_tot_disc = order_grp["discount"].transform("sum")
df["order_discount_share"] = (df["discount"] / (order_tot_disc + 1e-5)).astype(np.float32)
df["price_rank_in_order"] = order_grp["unit_price"].rank(pct=True).astype(np.float32)
order_price_mean = order_grp["unit_price"].transform("mean")
order_price_std = order_grp["unit_price"].transform("std").replace(0, 1.0).fillna(1.0)
df["item_price_zscore_in_order"] = ((df["unit_price"] - order_price_mean) / order_price_std).astype(np.float32)

df["n_same_product_in_order"] = df.groupby(["order_id", "product_id"])["line_no"].transform("count").astype(np.int8)
df["n_same_category_in_order"] = df.groupby(["order_id", "category_clean"])["line_no"].transform("count").astype(np.int8)
df["is_multi_variant_order"] = (df["n_same_category_in_order"] > 1).astype(np.int8)

cust_grp = df.groupby("customer_id")
df["cust_weight_diff"] = (df["weight_kg"] - cust_grp["weight_kg"].transform("mean")).astype(np.float32)
df["cust_discount_diff"] = (df["discount"] - cust_grp["discount"].transform("mean")).astype(np.float32)
df["cust_cart_size_ratio"] = (df["items_in_order"] / (cust_grp["items_in_order"].transform("mean") + 1e-3)).astype(np.float32)
df["cust_price_ratio"] = (df["unit_price"] / (cust_grp["unit_price"].transform("mean") + 1e-3)).astype(np.float32)

cat_grp = df.groupby("category_clean")
df["weight_rel_cat"] = (df["weight_kg"] / (cat_grp["weight_kg"].transform("mean") + 1e-3)).astype(np.float32)
df["price_rel_cat"] = (df["unit_price"] / (cat_grp["unit_price"].transform("mean") + 1e-3)).astype(np.float32)
df["discount_diff_cat"] = (df["discount"] - cat_grp["discount"].transform("mean")).astype(np.float32)

print(">>> Computing customer sequence timing dynamics...")
df.sort_values(by=["customer_id", "category_clean", "order_ts"], inplace=True)
prev_cat = df.groupby(["customer_id", "category_clean"])["order_ts"].shift(1)
next_cat = df.groupby(["customer_id", "category_clean"])["order_ts"].shift(-1)
df["cust_cat_days_since_prev"] = ((df["order_ts"] - prev_cat).dt.total_seconds() / 86400.0).fillna(999.0).astype(np.float32)
df["cust_cat_days_until_next"] = ((next_cat - df["order_ts"]).dt.total_seconds() / 86400.0).fillna(999.0).astype(np.float32)

df.sort_values(by=["customer_id", "product_id", "order_ts"], inplace=True)
prev_prod = df.groupby(["customer_id", "product_id"])["order_ts"].shift(1)
next_prod = df.groupby(["customer_id", "product_id"])["order_ts"].shift(-1)
df["cust_prod_days_since_prev"] = ((df["order_ts"] - prev_prod).dt.total_seconds() / 86400.0).fillna(999.0).astype(np.float32)
df["cust_prod_days_until_next"] = ((next_prod - df["order_ts"]).dt.total_seconds() / 86400.0).fillna(999.0).astype(np.float32)
del prev_cat, next_cat, prev_prod, next_prod
gc.collect()

df["store_x_region"] = df["store_id"].astype(str) + "_" + df["region_clean"].astype(str)
df["route_x_format"] = df["region_clean"].astype(str) + "_" + df["region_clean_store"].astype(str) + "_" + df["format"].astype(str)
df["store_x_cat"] = df["store_id"].astype(str) + "_" + df["category_clean"].astype(str)
df["supp_x_cat"] = df["supplier_code"].astype(str) + "_" + df["category_clean"].astype(str)
df["ctype_x_loyalty"] = df["customer_type"].astype(str) + "_" + df["loyalty_tier"].astype(str)
df["banner_id"] = df["banner_id"].astype(str)

df.sort_index(inplace=True)
print(f"Data prepared in {time.time() - t0:.1f}s. Total rows: {len(df)}")

# HIERARCHICAL LOG-ODDS
print("=" * 80)
print(">>> [2/6] COMPUTING LAG-ALIGNED HIERARCHICAL LOG-ODDS TOWER...")

def compute_hierarchical_log_odds(hist_df, target_df):
    res = pd.DataFrame(index=target_df.index)
    global_p0 = hist_df["returned"].mean()
    if np.isnan(global_p0):
        global_p0 = GLOBAL_PRIOR

    cat_stats = hist_df.groupby("category_clean")["returned"].agg(["count", "mean"])
    c_prob = (cat_stats["count"] * cat_stats["mean"] + 50.0 * global_p0) / (cat_stats["count"] + 50.0)
    cat_te = target_df["category_clean"].map(c_prob).fillna(global_p0).values
    res["category_clean_te"] = cat_te.astype(np.float32)

    st_stats = hist_df.groupby("store_id")["returned"].agg(["count", "mean"])
    st_prob = (st_stats["count"] * st_stats["mean"] + 50.0 * global_p0) / (st_stats["count"] + 50.0)
    store_te = target_df["store_id"].map(st_prob).fillna(global_p0).values
    res["store_id_te"] = store_te.astype(np.float32)
    res["store_id_lo"] = to_log_odds_delta(store_te, global_p0)

    sr_stats = hist_df.groupby("store_x_region")["returned"].agg(["count", "mean"])
    sr_merged = target_df[["store_x_region"]].merge(sr_stats, on="store_x_region", how="left")
    sr_cnt = sr_merged["count"].fillna(0).values
    sr_mn = sr_merged["mean"].fillna(0).values
    p_sr = (sr_cnt * sr_mn + 35.0 * store_te) / (sr_cnt + 35.0)
    res["store_x_region_lo"] = to_log_odds_delta(p_sr, global_p0)

    p_stats = hist_df.groupby("product_id")["returned"].agg(["count", "mean"])
    p_merged = target_df[["product_id"]].merge(p_stats, on="product_id", how="left")
    p_cnt = p_merged["count"].fillna(0).values
    p_mn = p_merged["mean"].fillna(0).values
    p_prod = (p_cnt * p_mn + 35.0 * cat_te) / (p_cnt + 35.0)
    res["product_id_te"] = p_prod.astype(np.float32)
    res["product_id_lo"] = to_log_odds_delta(p_prod, global_p0)

    reg_mean = hist_df.groupby("same_region")["returned"].mean().to_dict()
    sr_prior = target_df["same_region"].map(reg_mean).fillna(global_p0).values
    rf_stats = hist_df.groupby("route_x_format")["returned"].agg(["count", "mean"])
    rf_merged = target_df[["route_x_format"]].merge(rf_stats, on="route_x_format", how="left")
    rf_cnt = rf_merged["count"].fillna(0).values
    rf_mn = rf_merged["mean"].fillna(0).values
    p_rf = (rf_cnt * rf_mn + 40.0 * sr_prior) / (rf_cnt + 40.0)
    res["route_x_format_te"] = p_rf.astype(np.float32)
    res["route_x_format_lo"] = to_log_odds_delta(p_rf, global_p0)

    cp_stats = hist_df.groupby(["customer_id", "product_id"])["returned"].agg(["count", "mean"])
    cp_merged = target_df[["customer_id", "product_id"]].merge(cp_stats, on=["customer_id", "product_id"], how="left")
    cp_cnt = cp_merged["count"].fillna(0).values
    cp_mn = cp_merged["mean"].fillna(0).values
    p_cp = (cp_cnt * cp_mn + 8.0 * p_prod) / (cp_cnt + 8.0)
    res["cust_prod_prev_lo"] = to_log_odds_delta(p_cp, global_p0)

    sc_stats = hist_df.groupby("store_x_cat")["returned"].agg(["count", "mean"])
    sc_merged = target_df[["store_x_cat"]].merge(sc_stats, on="store_x_cat", how="left")
    sc_cnt = sc_merged["count"].fillna(0).values
    sc_mn = sc_merged["mean"].fillna(0).values
    p_sc = (sc_cnt * sc_mn + 40.0 * store_te) / (sc_cnt + 40.0)
    res["store_x_cat_lo"] = to_log_odds_delta(p_sc, global_p0)

    hist_supp = hist_df[hist_df["month_idx"] >= 4]
    if len(hist_supp) < 1000:
        hist_supp = hist_df
    s_stats = hist_supp.groupby("supplier_code")["returned"].agg(["count", "mean"])
    s_merged = target_df[["supplier_code"]].merge(s_stats, on="supplier_code", how="left")
    s_cnt = s_merged["count"].fillna(0).values
    s_mn = s_merged["mean"].fillna(0).values
    p_supp = (s_cnt * s_mn + 40.0 * global_p0) / (s_cnt + 40.0)
    res["supplier_code_lo"] = to_log_odds_delta(p_supp, global_p0)

    spc_stats = hist_supp.groupby("supp_x_cat")["returned"].agg(["count", "mean"])
    spc_merged = target_df[["supp_x_cat"]].merge(spc_stats, on="supp_x_cat", how="left")
    spc_cnt = spc_merged["count"].fillna(0).values
    spc_mn = spc_merged["mean"].fillna(0).values
    p_spc = (spc_cnt * spc_mn + 40.0 * p_supp) / (spc_cnt + 40.0)
    res["supp_x_cat_lo"] = to_log_odds_delta(p_spc, global_p0)

    cl_stats = hist_df.groupby("ctype_x_loyalty")["returned"].agg(["count", "mean"])
    cl_prob = (cl_stats["count"] * cl_stats["mean"] + 30.0 * global_p0) / (cl_stats["count"] + 30.0)
    cl_prior = target_df["ctype_x_loyalty"].map(cl_prob).fillna(global_p0).values

    c_stats = hist_df.groupby("customer_id")["returned"].agg(["count", "mean"])
    c_merged = target_df[["customer_id"]].merge(c_stats, on="customer_id", how="left")
    c_cnt = c_merged["count"].fillna(0).values
    c_mn = c_merged["mean"].fillna(0).values
    p_cust = (c_cnt * c_mn + 30.0 * cl_prior) / (c_cnt + 30.0)
    res["customer_id_te"] = p_cust.astype(np.float32)
    res["customer_id_lo"] = to_log_odds_delta(p_cust, global_p0)

    cc_stats = hist_df.groupby(["customer_id", "category_clean"])["returned"].agg(["count", "mean"])
    cc_merged = target_df[["customer_id", "category_clean"]].merge(cc_stats, on=["customer_id", "category_clean"], how="left")
    cc_cnt = cc_merged["count"].fillna(0).values
    cc_mn = cc_merged["mean"].fillna(0).values
    p_cc = (cc_cnt * cc_mn + 12.0 * p_cust) / (cc_cnt + 12.0)
    res["cust_cat_prev_lo"] = to_log_odds_delta(p_cc, global_p0)

    # banner_id entity
    b_stats = hist_df.groupby("banner_id")["returned"].agg(["count", "mean"])
    b_merged = target_df[["banner_id"]].merge(b_stats, on="banner_id", how="left")
    b_cnt = b_merged["count"].fillna(0).values
    b_mn = b_merged["mean"].fillna(0).values
    p_b = (b_cnt * b_mn + 20.0 * global_p0) / (b_cnt + 20.0)
    res["banner_id_lo"] = to_log_odds_delta(p_b, global_p0)

    # Cart Peer LO
    temp = pd.DataFrame({"order_id": target_df["order_id"],
                         "p_lo": res["product_id_lo"], "sc_lo": res["store_x_cat_lo"]})
    order_p_sum = temp.groupby("order_id")["p_lo"].transform("sum")
    order_sc_sum = temp.groupby("order_id")["sc_lo"].transform("sum")
    order_n = temp.groupby("order_id")["p_lo"].transform("count")
    res["loo_order_prod_lo"] = np.where(order_n > 1, (order_p_sum - temp["p_lo"]) / (order_n - 1), 0.0).astype(np.float32)
    res["loo_order_store_cat_lo"] = np.where(order_n > 1, (order_sc_sum - temp["sc_lo"]) / (order_n - 1), 0.0).astype(np.float32)

    return res

# Slices
hist_s1 = df[(df["is_test"] == 0) & (df["month_idx"] <= 6)]
targ_s1 = df[(df["is_test"] == 0) & (df["month_idx"] >= 7) & (df["month_idx"] <= 12)]
lo_s1 = compute_hierarchical_log_odds(hist_s1, targ_s1)

hist_s2 = df[(df["is_test"] == 0) & (df["month_idx"] <= 12)]
targ_s2 = df[(df["is_test"] == 0) & (df["month_idx"] >= 13) & (df["month_idx"] <= 18)]
lo_s2 = compute_hierarchical_log_odds(hist_s2, targ_s2)

hist_s3 = df[(df["is_test"] == 0) & (df["month_idx"] <= 18)]
targ_s3 = df[(df["is_test"] == 0) & (df["month_idx"] >= 19) & (df["month_idx"] <= 24)]
lo_s3 = compute_hierarchical_log_odds(hist_s3, targ_s3)

hist_test = df[(df["is_test"] == 0) & (df["month_idx"] <= 24)]
targ_test = df[df["is_test"] == 1]
lo_test = compute_hierarchical_log_odds(hist_test, targ_test)

hist_m1_3 = df[(df["is_test"] == 0) & (df["month_idx"] <= 3)]
targ_m4_6 = df[(df["is_test"] == 0) & (df["month_idx"] >= 4) & (df["month_idx"] <= 6)]
lo_m4_6 = compute_hierarchical_log_odds(hist_m1_3, targ_m4_6)

hist_m4_6 = df[(df["is_test"] == 0) & (df["month_idx"] >= 4) & (df["month_idx"] <= 6)]
targ_m1_3 = df[(df["is_test"] == 0) & (df["month_idx"] <= 3)]
lo_m1_3 = compute_hierarchical_log_odds(hist_m4_6, targ_m1_3)
lo_m1_6 = pd.concat([lo_m1_3, lo_m4_6]).sort_index()

lo_all = pd.concat([lo_m1_6, lo_s1, lo_s2, lo_s3, lo_test]).sort_index()
for col in lo_all.columns:
    df[col] = lo_all[col]

del hist_s1, hist_s2, hist_s3, hist_test, targ_s1, targ_s2, targ_s3, targ_test
del lo_s1, lo_s2, lo_s3, lo_test, lo_m1_3, lo_m4_6, lo_m1_6, lo_all
gc.collect()

# FEATURE SETS
cat_cols_tower1 = [
    "channel", "payment_method", "customer_type", "loyalty_tier",
    "category_clean", "format", "region_clean", "region_clean_store", "ctype_x_loyalty",
    "age_band", "preferred_contact", "packaging_type", "color"
]
for c in cat_cols_tower1:
    df[c] = df[c].fillna("missing").astype(str).astype("category")

tower1_features = [
    "items_in_order_pct_m", "line_no_pct_m", "quantity_pct_m", "order_value_pct_m",
    "cust_tenure_pct_m", "prod_tenure_pct_m", "clicks_per_item_pct_m",
    "items_in_order_capped", "line_no_capped", "line_ratio",
    "weight_kg", "prod_volume", "prod_density", "discount", "has_discount", "is_free_freight",
    "price_ratio", "freight_cost", "freight_ratio", "item_share_order", "qty_share_order",
    "order_discount_share", "price_rank_in_order", "item_price_zscore_in_order",
    "return_window_days", "session_clicks", "seller_rating", "manager_tenure_years", "same_region",
    "n_same_product_in_order", "n_same_category_in_order", "is_multi_variant_order",
    "order_mean_weight", "order_max_weight", "cust_weight_diff", "cust_discount_diff",
    "cust_cart_size_ratio", "cust_price_ratio", "weight_rel_cat", "price_rel_cat", "discount_diff_cat",
    "cust_cat_days_since_prev", "cust_cat_days_until_next", "cust_prod_days_since_prev", "cust_prod_days_until_next",
    "store_id_te", "route_x_format_te", "category_clean_te",
    "review_count", "size_sqm", "raw_store_tenure", "newsletter_opt_in",
    "hour", "dayofweek", "is_weekend",
    "holiday_flag", "cpi", "fuel_price", "temperature_f", "unemployment", "fuel_surcharge",
    *cat_cols_tower1
]

log_odds_features = [
    "store_x_region_lo", "product_id_lo", "route_x_format_lo", "store_id_lo",
    "cust_prod_prev_lo", "store_x_cat_lo", "supp_x_cat_lo", "supplier_code_lo",
    "customer_id_lo", "cust_cat_prev_lo", "loo_order_prod_lo", "loo_order_store_cat_lo",
    "banner_id_lo"
]
df[log_odds_features] = df[log_odds_features].fillna(0.0).astype(np.float32)

# VALIDATION
print("=" * 80)
print(">>> [3/6] OUT-OF-TIME VALIDATION...")
val_mask = (df["is_test"] == 0) & (df["month_idx"] >= 19) & (df["month_idx"] <= 24)
tr_val_mask = (df["is_test"] == 0) & (df["month_idx"] >= 7) & (df["month_idx"] <= 18)

y_val = df.loc[val_mask, "returned"].values.astype(np.float32)
y_tr_val = df.loc[tr_val_mask, "returned"].values.astype(np.float32)
val_meta = df.loc[val_mask, ["order_id", "year_month", "items_in_order"]].copy()

def make_dmatrix(features, mask, label=None):
    X = df.loc[mask, features]
    d = xgb.DMatrix(X, label=label, enable_categorical=True)
    return d

xgb_t1_params = {
    "objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist", "device": "cuda",
    "learning_rate": 0.04, "max_depth": 6, "min_child_weight": 150, "colsample_bytree": 0.75,
    "subsample": 0.80, "reg_alpha": 3.0, "reg_lambda": 10.0, "max_bin": 256, "random_state": 42,
}
xgb_t3_params = {
    "objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist", "device": "cuda",
    "learning_rate": 0.03, "max_depth": 5, "min_child_weight": 250, "colsample_bytree": 0.50,
    "subsample": 0.80, "reg_alpha": 10.0, "reg_lambda": 25.0, "max_bin": 256, "random_state": 42,
}

def train_xgb(features, params, tr_mask, va_mask, num_round=800, early=40, cat_cols=None):
    dtr = make_dmatrix(features, tr_mask, df.loc[tr_mask, "returned"].values)
    dva = make_dmatrix(features, va_mask, df.loc[va_mask, "returned"].values)
    booster = xgb.train(params, dtr, num_boost_round=num_round,
                        evals=[(dtr, "train"), (dva, "val")],
                        callbacks=[xgb.callback.EarlyStopping(rounds=early, metric_name="auc", maximize=True, save_best=True)],
                        verbose_eval=200)
    best = getattr(booster, "best_iteration", None)
    if best is None:
        best = num_round
    pred = booster.predict(dva, iteration_range=(0, best + 1))
    return booster, pred, best

print("\n--- Tower 1: Structural XGBoost (GPU) ---")
m_t1, p_val_t1, best_t1 = train_xgb(tower1_features, xgb_t1_params, tr_val_mask, val_mask)
auc_t1 = roc_auc_score(y_val, p_val_t1)
print(f"--> Tower1 AUC {auc_t1:.5f} best_iter {best_t1}")

print("\n--- Tower 2: Polynomial Log-Odds Logistic Regression ---")
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
scaler = StandardScaler()
lr = LogisticRegression(C=0.5, solver="lbfgs", max_iter=500, random_state=42)
tower2_pipe = Pipeline([("poly", poly), ("scaler", scaler), ("lr", lr)])
tower2_pipe.fit(df.loc[tr_val_mask, log_odds_features], y_tr_val)
p_val_t2 = tower2_pipe.predict_proba(df.loc[val_mask, log_odds_features])[:, 1]
auc_t2 = roc_auc_score(y_val, p_val_t2)
print(f"--> Tower2 AUC {auc_t2:.5f}")

df["tower2_oot_log_odds"] = tower2_pipe.decision_function(df[log_odds_features]).astype(np.float32)

print("\n--- Tower 3: Full-Feature XGBoost (GPU) ---")
tower3_features = tower1_features + ["tower2_oot_log_odds"] + log_odds_features
m_t3, p_val_t3, best_t3 = train_xgb(tower3_features, xgb_t3_params, tr_val_mask, val_mask)
auc_t3 = roc_auc_score(y_val, p_val_t3)
print(f"--> Tower3 AUC {auc_t3:.5f} best_iter {best_t3}")

print("\n--- Tower 4: CatBoost (GPU) ---")
cat_features_idx = [tower3_features.index(c) for c in cat_cols_tower1 if c in tower3_features]
train_pool = Pool(df.loc[tr_val_mask, tower3_features], label=y_tr_val, cat_features=cat_features_idx)
val_pool = Pool(df.loc[val_mask, tower3_features], label=y_val, cat_features=cat_features_idx)
cb = CatBoostClassifier(iterations=600, learning_rate=0.03, depth=6, l2_leaf_reg=10.0,
                        task_type="GPU", devices="0", random_seed=42, verbose=200, eval_metric="AUC")
cb.fit(train_pool, eval_set=val_pool, early_stopping_rounds=40, use_best_model=True)
p_val_t4 = cb.predict_proba(val_pool)[:, 1]
auc_t4 = roc_auc_score(y_val, p_val_t4)
print(f"--> Tower4 AUC {auc_t4:.5f}")

np.savez_compressed(f"{OUTPUT_DIR}/val_preds.npz", y=y_val, t1=p_val_t1, t2=p_val_t2, t3=p_val_t3, t4=p_val_t4)

print("\n--- Optimizing Meta-Blend ---")
z1 = logit(np.clip(p_val_t1, EPS, 1 - EPS))
z2 = logit(np.clip(p_val_t2, EPS, 1 - EPS))
z3 = logit(np.clip(p_val_t3, EPS, 1 - EPS))
z4 = logit(np.clip(p_val_t4, EPS, 1 - EPS))
Z = np.column_stack([z1, z2, z3, z4])

def neg_auc(w):
    w = np.clip(w, 0, None)
    s = w.sum()
    if s == 0:
        return 0.0
    w = w / s
    z = Z @ w
    return -roc_auc_score(y_val, z)

res = minimize(neg_auc, x0=np.array([0.2, 0.35, 0.25, 0.2]), method="SLSQP",
               bounds=[(0, 1)] * 4, constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
w = np.clip(res.x, 0, None); w = w / w.sum()
print(f"--> Weights: T1={w[0]:.3f} T2={w[1]:.3f} T3={w[2]:.3f} T4={w[3]:.3f}")
z_blend_val = Z @ w
best_blend_auc = roc_auc_score(y_val, z_blend_val)
print(f"--> Meta-Blend AUC: {best_blend_auc:.5f}")

val_meta["z_blend"] = z_blend_val
order_z_mean = val_meta.groupby("order_id")["z_blend"].transform("mean")
multi_item_mask = (val_meta["items_in_order"] >= 2).values

best_alpha = 0.20
best_smoothed_auc = best_blend_auc
for alpha in np.linspace(0.05, 0.30, 11):
    z_smooth = z_blend_val.copy()
    z_smooth[multi_item_mask] = (1.0 - alpha) * z_blend_val[multi_item_mask] + alpha * order_z_mean.values[multi_item_mask]
    score = roc_auc_score(y_val, z_smooth)
    if score > best_smoothed_auc:
        best_smoothed_auc = score
        best_alpha = alpha
print(f"--> + Order Smoothing AUC: {best_smoothed_auc:.5f} (alpha={best_alpha:.2f})")

del m_t1, m_t3, train_pool, val_pool
gc.collect()

# FULL-DATA REFIT
print("=" * 80)
print(">>> [4/6] REFITTING ON FULL DATA...")
full_train_mask = (df["is_test"] == 0)
test_mask = (df["is_test"] == 1)
y_full = df.loc[full_train_mask, "returned"].values.astype(np.float32)
test_meta = df.loc[test_mask, ["order_line_id", "order_id", "year_month", "items_in_order"]].copy()

print("Refitting Tower1 (3 seeds)...")
t1_preds = np.zeros(test_mask.sum(), dtype=np.float32)
for seed in [42, 101, 777]:
    params = dict(xgb_t1_params); params["random_state"] = seed
    dtr = xgb.DMatrix(df.loc[full_train_mask, tower1_features], label=y_full, enable_categorical=True)
    booster = xgb.train(params, dtr, num_boost_round=best_t1 + 1, verbose_eval=0)
    dte = xgb.DMatrix(df.loc[test_mask, tower1_features], enable_categorical=True)
    t1_preds += booster.predict(dte, iteration_range=(0, best_t1 + 1)) / 3.0
    del dtr, dte, booster; gc.collect()

print("Refitting Tower2 (poly LR)...")
tower2_pipe_full = Pipeline([("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
                              ("scaler", StandardScaler()),
                              ("lr", LogisticRegression(C=0.5, solver="lbfgs", max_iter=500, random_state=42))])
tower2_pipe_full.fit(df.loc[full_train_mask, log_odds_features], y_full)
t2_preds = tower2_pipe_full.predict_proba(df.loc[test_mask, log_odds_features])[:, 1]
df.loc[full_train_mask, "tower2_oot_log_odds"] = tower2_pipe_full.decision_function(df.loc[full_train_mask, log_odds_features]).astype(np.float32)
df.loc[test_mask, "tower2_oot_log_odds"] = tower2_pipe_full.decision_function(df.loc[test_mask, log_odds_features]).astype(np.float32)

print("Refitting Tower3 (3 seeds)...")
t3_preds = np.zeros(test_mask.sum(), dtype=np.float32)
tower3_features = tower1_features + ["tower2_oot_log_odds"] + log_odds_features
for seed in [42, 101, 777]:
    params = dict(xgb_t3_params); params["random_state"] = seed
    dtr = xgb.DMatrix(df.loc[full_train_mask, tower3_features], label=y_full, enable_categorical=True)
    booster = xgb.train(params, dtr, num_boost_round=best_t3 + 1, verbose_eval=0)
    dte = xgb.DMatrix(df.loc[test_mask, tower3_features], enable_categorical=True)
    t3_preds += booster.predict(dte, iteration_range=(0, best_t3 + 1)) / 3.0
    del dtr, dte, booster; gc.collect()

print("Refitting Tower4 (CatBoost)...")
cat_features_idx = [tower3_features.index(c) for c in cat_cols_tower1 if c in tower3_features]
full_pool = Pool(df.loc[full_train_mask, tower3_features], label=y_full, cat_features=cat_features_idx)
test_pool = Pool(df.loc[test_mask, tower3_features], cat_features=cat_features_idx)
cb_full = CatBoostClassifier(iterations=cb.best_iteration_ + 1, learning_rate=0.03, depth=6, l2_leaf_reg=10.0,
                             task_type="GPU", devices="0", random_seed=42, verbose=0, eval_metric="AUC")
cb_full.fit(full_pool, verbose=0)
t4_preds = cb_full.predict_proba(test_pool)[:, 1]
del full_pool, test_pool, cb_full; gc.collect()

# META-COMBINATION & CALIBRATION
print("=" * 80)
print(">>> [5/6] FINAL SUBMISSIONS...")
z_test = np.column_stack([
    logit(np.clip(t1_preds, EPS, 1 - EPS)),
    logit(np.clip(t2_preds, EPS, 1 - EPS)),
    logit(np.clip(t3_preds, EPS, 1 - EPS)),
    logit(np.clip(t4_preds, EPS, 1 - EPS)),
])
z_blend_full = z_test @ w

def process_and_calibrate(z_raw, meta_df, alpha=best_alpha):
    meta = meta_df.copy()
    meta["z"] = z_raw
    multi_mask = (meta["items_in_order"] >= 2).values
    order_mean = meta.groupby("order_id")["z"].transform("mean").values
    z_smooth = meta["z"].values.copy()
    z_smooth[multi_mask] = (1.0 - alpha) * z_smooth[multi_mask] + alpha * order_mean[multi_mask]
    meta["z_smooth"] = z_smooth
    m_grp = meta.groupby("year_month")["z_smooth"]
    m_mean = m_grp.transform("mean").values
    m_std = m_grp.transform("std").replace(0, 1.0).fillna(1.0).values
    meta["z_final"] = (meta["z_smooth"] - m_mean) / m_std
    p_final = expit(meta["z_final"].values + logit(GLOBAL_PRIOR))
    return p_final

np.savez_compressed(f"{OUTPUT_DIR}/test_preds.npz", t1=t1_preds, t2=t2_preds, t3=t3_preds, t4=t4_preds, w=w)

z_primary = 0.20 * z_test[:, 0] + 0.80 * z_test[:, 2]
sub_primary_preds = process_and_calibrate(z_primary, test_meta, alpha=best_alpha)
sub_primary = pd.DataFrame({"order_line_id": test_meta["order_line_id"], "returned": sub_primary_preds})
sub_primary.to_csv(f"{OUTPUT_DIR}/submission_primary.csv", index=False)
print(f"--> Saved primary (T1+T3 blend) submission ({len(sub_primary)} rows)")

sub_blend_preds = process_and_calibrate(z_blend_full, test_meta, alpha=best_alpha)
sub_blend = pd.DataFrame({"order_line_id": test_meta["order_line_id"], "returned": sub_blend_preds})
sub_blend.to_csv(f"{OUTPUT_DIR}/submission_blend.csv", index=False)
print(f"--> Saved 4-tower blend submission ({len(sub_blend)} rows)")

z_safe = 0.4 * z_test[:, 0] + 0.6 * z_test[:, 1]
sub_safe_preds = process_and_calibrate(z_safe, test_meta, alpha=best_alpha)
sub2 = pd.DataFrame({"order_line_id": test_meta["order_line_id"], "returned": sub_safe_preds})
sub2.to_csv(f"{OUTPUT_DIR}/submission_tower12_safe.csv", index=False)
print(f"--> Saved safe baseline ({len(sub2)} rows)")

test_meta["sub_pred"] = sub_primary_preds
monthly_means = test_meta.groupby("year_month")["sub_pred"].agg(["count", "mean", "std"])
print("\n--- MONTHLY PREDICTION DIAGNOSTICS ---")
print(monthly_means)

total_time = time.time() - t0
print("=" * 80)
print(f"PIPELINE COMPLETED IN {total_time:.1f}s ({total_time/60.0:.2f} mins)")
print(f"Tower1 AUC {auc_t1:.5f} | Tower2 {auc_t2:.5f} | Tower3 {auc_t3:.5f} | Tower4 {auc_t4:.5f}")
print(f"Meta-Blend {best_blend_auc:.5f} | +Smoothing {best_smoothed_auc:.5f}")
print("=" * 80)

>>> [1/6] LOADING DATA, HYGIENE & POINT-IN-TIME MERGES...
>>> Computing monthly percentile ranks...
>>> Computing customer sequence timing dynamics...
Data prepared in 161.4s. Total rows: 1880000
>>> [2/6] COMPUTING LAG-ALIGNED HIERARCHICAL LOG-ODDS TOWER...
>>> [3/6] OUT-OF-TIME VALIDATION...

--- Tower 1: Structural XGBoost (GPU) ---
[0]	train-auc:0.64336	val-auc:0.63869
[193]	train-auc:0.70722	val-auc:0.67341
--> Tower1 AUC 0.67357 best_iter 154

--- Tower 2: Polynomial Log-Odds Logistic Regression ---
--> Tower2 AUC 0.65734

--- Tower 3: Full-Feature XGBoost (GPU) ---
[0]	train-auc:0.64877	val-auc:0.64795
[200]	train-auc:0.70400	val-auc:0.68764
[207]	train-auc:0.70472	val-auc:0.68748
--> Tower3 AUC 0.68766 best_iter 168

--- Tower 4: CatBoost (GPU) ---


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6596956	best: 0.6596956 (0)	total: 126ms	remaining: 1m 15s
200:	test: 0.6852966	best: 0.6855439 (188)	total: 13.6s	remaining: 27.1s
bestTest = 0.6855438948
bestIteration = 188
Shrink model to first 189 iterations.
--> Tower4 AUC 0.68554

--- Optimizing Meta-Blend ---
--> Weights: T1=0.227 T2=0.003 T3=0.352 T4=0.418
--> Meta-Blend AUC: 0.68787
--> + Order Smoothing AUC: 0.68799 (alpha=0.10)
>>> [4/6] REFITTING ON FULL DATA...
Refitting Tower1 (3 seeds)...
Refitting Tower2 (poly LR)...
Refitting Tower3 (3 seeds)...
Refitting Tower4 (CatBoost)...


Default metric period is 5 because AUC is/are not implemented for GPU


>>> [5/6] FINAL SUBMISSIONS...
--> Saved primary (T1+T3 blend) submission (479995 rows)
--> Saved 4-tower blend submission (479995 rows)
--> Saved safe baseline (479995 rows)

--- MONTHLY PREDICTION DIAGNOSTICS ---
            count      mean       std
year_month                           
2024-12        26  0.218892  0.137041
2025-01     73289  0.219949  0.158420
2025-02     59545  0.219844  0.156171
2025-03     75724  0.219814  0.157427
2025-04     78256  0.219753  0.157331
2025-05     90260  0.219670  0.160348
2025-06     95784  0.219487  0.162212
2025-07      7111  0.219047  0.159740
PIPELINE COMPLETED IN 359.0s (5.98 mins)
Tower1 AUC 0.67357 | Tower2 0.65734 | Tower3 0.68766 | Tower4 0.68554
Meta-Blend 0.68787 | +Smoothing 0.68799
